# HidroVision AI — Modelo Preditivo de Nível do Rio

**Equipe 10 | FETIN 2026 — INATEL**

Treinamento de três modelos XGBoost que preveem o nível do Rio Sapucaí em
**t+6h, t+12h e t+24h**, a partir de dados reais de duas fontes oficiais:

| Fonte | Dado | Estação |
|---|---|---|
| ANA / HidroWebService | nível do rio (cm) | 61305000 — Santa Rita do Sapucaí |
| INMET | chuva horária (mm) | A531 — Maria da Fé |

Período: janeiro de 2023 a agosto de 2026 (~27.000 horas). O modelo aprende a
relação entre chuva e nível a partir desse histórico.

## Alvo: a variação.

Os modelos preveem a **variação** do nível (`nível(t+h) − nível(t)`); o nível
previsto é `nível atual + variação`.

A escolha é determinante. Prevendo o nível absoluto, a resposta está quase toda
contida no nível atual — o modelo aprende a copiá-lo e nunca precisa consultar a
chuva. Prevendo a variação, ele é obrigado a usar as demais variáveis, e a chuva
passa a pesar entre 14% e 41% nas decisões (medido na seção 5).

## Validação

**Temporal**: treino de 2023 a 2025, teste em 2026. Split aleatório em série
temporal permitiria ao modelo ver o futuro e inflaria as métricas.

**Critério**: superar o *baseline de persistência* — prever que o nível daqui a
N horas será igual ao atual. É exigente, porque em rio calmo o baseline acerta
quase sempre.

## 1. Setup

In [ ]:
!pip install -q xgboost

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

print("xgboost", xgb.__version__)

## 2. Carregar o dataset

Carregar os dados para o treinamento.

In [ ]:
# No Colab: envie o dados_treino.csv quando o botão aparecer.
from google.colab import files
enviados = files.upload()
ARQUIVO = list(enviados.keys())[0]

# Localmente: comente as três linhas acima e use o caminho direto.
# ARQUIVO = "dados_treino.csv"

print("arquivo:", ARQUIVO)

In [ ]:
FUSO = "America/Sao_Paulo"
HORIZONTES = (6, 12, 24)
ANO_TESTE = 2026

def carregar(caminho):
    df = pd.read_csv(caminho, parse_dates=["datahora"])
    df["datahora"] = pd.to_datetime(df["datahora"], utc=True).dt.tz_convert(FUSO)
    df = df.drop_duplicates("datahora").sort_values("datahora").set_index("datahora")
    # grade horária contínua: sem isso um lag de 1 posição pode virar lag de 10 h
    # onde a telemetria falhou
    grade = pd.date_range(df.index.min(), df.index.max(), freq="1h", tz=FUSO)
    df = df.reindex(grade); df.index.name = "datahora"
    return df

dados = carregar(ARQUIVO)
print(f"{len(dados)} horas | {dados.index.min()} -> {dados.index.max()}")
print(f"cobertura nível: {dados['nivel_cm'].notna().mean()*100:.1f}%")

## 3. Features e alvo

São 23 variáveis, todas olhando apenas para trás: estado e histórico do nível,
velocidade de variação, chuva (intensidade e acumulados em várias janelas),
a interação entre chuva acumulada e nível, a mesma chuva sobre solo saturado
eleva o rio muito mais e a sazonalidade do mês.

O alvo `nivel_t{h}` é a **variação** em centímetros, não o nível absoluto.

In [ ]:
def construir(df, horizontes=HORIZONTES):
    X = pd.DataFrame(index=df.index)
    nivel = df["nivel_cm"]
    chuva = df["chuva_mm"].fillna(0.0)

    # estado atual e histórico do nível
    X["nivel"] = nivel
    for h in (1, 2, 3, 6, 12, 24):
        X[f"nivel_lag{h}"] = nivel.shift(h)
    X["delta_1h"] = nivel - nivel.shift(1)
    X["delta_3h"] = nivel - nivel.shift(3)
    X["delta_6h"] = nivel - nivel.shift(6)
    X["tend_6h"] = X["delta_6h"] / 6.0

    # chuva: intensidade atual e acumulados (o slider atua justamente aqui)
    X["chuva"] = chuva
    for j in (3, 6, 12, 24, 48, 72):
        X[f"chuva_acum{j}"] = chuva.rolling(j, min_periods=1).sum()
    X["chuva_max6"] = chuva.rolling(6, min_periods=1).max()
    houve = (chuva > 1.0).astype(int); g = houve.cumsum()
    X["horas_sem_chuva"] = (houve.groupby(g).cumcount()
                            .where(g > 0, 72).clip(upper=72))
    # chuva sobre solo saturado sobe muito mais que sobre solo seco
    X["chuva24_x_nivel"] = X["chuva_acum24"] * X["nivel"] / 100.0

    mes = df.index.month
    X["mes_sen"] = np.sin(2*np.pi*mes/12); X["mes_cos"] = np.cos(2*np.pi*mes/12)

    # ALVO = VARIAÇÃO (delta), não nível absoluto
    for h in horizontes:
        X[f"nivel_t{h}"] = nivel.shift(-h) - nivel
    return X

X = construir(dados)
FEATS = [c for c in X.columns if not c.startswith("nivel_t")]
print(f"{len(FEATS)} features")
print(X[[f"nivel_t{h}" for h in HORIZONTES]].describe().round(1))

In [ ]:
# checagem anti-vazamento
s = dados["nivel_cm"]
assert X["nivel_lag1"].equals(s.shift(1)), "lag inconsistente"
assert X["nivel_t6"].equals(s.shift(-6) - s), "alvo inconsistente"
assert not any(X[c].equals(s.shift(-6)) for c in FEATS), "VAZAMENTO de futuro!"
print("checagem de vazamento: OK")

def separar(X, horizonte, ano_teste=ANO_TESTE):
    alvo = f"nivel_t{horizonte}"
    d = X[FEATS + [alvo]].dropna()
    tr = d[d.index.year < ano_teste]; te = d[d.index.year >= ano_teste]
    return tr[FEATS], tr[alvo], te[FEATS], te[alvo]

for h in HORIZONTES:
    Xtr, ytr, Xte, yte = separar(X, h)
    print(f"t+{h}h -> treino {len(Xtr)} | teste {len(Xte)}")

## 4. Treino

Treinamento do modelo

In [ ]:
LIMIARES = {"atencao": 228, "alerta": 304, "emergencia": 388}

def treinar_um(horizonte):
    Xtr, ytr, Xte, yte = separar(X, horizonte)
    modelo = xgb.XGBRegressor(
        n_estimators=600, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
        reg_lambda=1.0, objective="reg:squarederror",
        early_stopping_rounds=50, eval_metric="mae",
        n_jobs=-1, random_state=42)
    corte = int(len(Xtr) * 0.85)          # validação também temporal
    modelo.fit(Xtr.iloc[:corte], ytr.iloc[:corte],
               eval_set=[(Xtr.iloc[corte:], ytr.iloc[corte:])], verbose=False)

    d_pred = modelo.predict(Xte)          # variação prevista
    d_real = yte.values                   # variação real
    mae = np.mean(np.abs(d_pred - d_real))
    mae_base = np.mean(np.abs(d_real))    # persistência = prever 0
    ganho = (1 - mae/mae_base) * 100

    print(f"\n{'='*60}\nt+{horizonte}h — treino {len(Xtr)} | teste {len(Xte)}")
    print(f"  MAE do modelo      {mae:6.2f} cm")
    print(f"  MAE da persistência {mae_base:6.2f} cm")
    print(f"  >>> ganho: {ganho:+.1f}%")

    for corte_cm in (5, 20):
        sel = np.abs(d_real) >= corte_cm
        if sel.sum() > 5:
            mm = np.mean(np.abs(d_pred[sel] - d_real[sel]))
            bb = np.mean(np.abs(d_real[sel]))
            print(f"  variações >={corte_cm}cm ({sel.sum()}h): modelo {mm:5.2f} | "
                  f"persist. {bb:5.2f} | ganho {(1-mm/bb)*100:+.0f}%")

    # nível absoluto reconstruído, para checar limiares
    nivel_prev = Xte["nivel"].values + d_pred
    nivel_real = Xte["nivel"].values + d_real
    for nome, lim in LIMIARES.items():
        real = nivel_real >= lim; prev = nivel_prev >= lim
        vp = (real & prev).sum(); fp = (~real & prev).sum(); fn = (real & ~prev).sum()
        if real.sum():
            prec = vp/(vp+fp) if vp+fp else np.nan
            rec = vp/(vp+fn) if vp+fn else np.nan
            print(f"  alerta '{nome}' (>{lim}cm): {int(real.sum())}h reais | "
                  f"precisão {prec:.2f} | recall {rec:.2f}")

    imp = pd.Series(modelo.feature_importances_, index=FEATS)
    grupos = {
        "nível/lags": imp[[c for c in FEATS if c.startswith("nivel")]].sum(),
        "deltas/tend": imp[[c for c in FEATS if "delta" in c or "tend" in c]].sum(),
        "chuva": imp[[c for c in FEATS if "chuva" in c or "horas_sem" in c]].sum(),
        "sazonal": imp[[c for c in FEATS if c.startswith("mes")]].sum()}
    print("  importância: " + " | ".join(f"{k} {v*100:.1f}%"
                                          for k, v in grupos.items()))

    modelo.save_model(f"modelo_delta_{horizonte}h.json")
    return modelo, {"horizonte": horizonte, "mae_modelo": mae,
                    "mae_baseline": mae_base, "ganho_pct": ganho,
                    "peso_chuva_pct": grupos["chuva"]*100}

modelos, linhas = {}, []
for h in HORIZONTES:
    mod, met = treinar_um(h)
    modelos[h] = mod; linhas.append(met)

metricas = pd.DataFrame(linhas)
metricas.to_csv("metricas_delta.csv", index=False)
metricas.round(2)

## 5. O peso da chuva nas decisões do modelo

A tabela abaixo compara a importância das variáveis de chuva sob os dois alvos
possíveis. A linha do alvo absoluto vem de um treino de controle feito sobre o
mesmo dataset, e é a razão pela qual este notebook usa o alvo em variação.

In [ ]:
comp = pd.DataFrame({
    "horizonte": ["t+6h", "t+12h", "t+24h"],
    "alvo absoluto (%)": [0.2, 0.7, 3.4],   # treino de controle, para comparação
    "alvo variação (%)": [metricas.loc[metricas.horizonte==h, "peso_chuva_pct"].values[0]
                          for h in HORIZONTES]})

fig, ax = plt.subplots(figsize=(8, 3.2))
xx = np.arange(3); w = 0.36
ax.bar(xx-w/2, comp["alvo absoluto (%)"], w, label="alvo: nível absoluto",
       color="#8FA8BF")
ax.bar(xx+w/2, comp["alvo variação (%)"], w, label="alvo: variação (delta)",
       color="#0B7FAB")
for i, v in enumerate(comp["alvo variação (%)"]):
    ax.annotate(f"{v:.0f}%", (i+w/2, v), xytext=(0, 3),
                textcoords="offset points", ha="center", fontweight="bold")
ax.set_xticks(xx); ax.set_xticklabels(comp["horizonte"])
ax.set_ylabel("peso da chuva na decisão (%)")
ax.set_title("Importância das features de chuva")
ax.legend(frameon=False, fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()
comp.round(1)

## 6. Simulação de chuva

Como a chuva pesa nas decisões, é possível responder a perguntas do tipo *"e se
chover 10 mm/h nas próximas horas?"*. A função abaixo recebe o estado atual do
rio e uma chuva hipotética, injeta esse valor nas variáveis de chuva acumulada e
devolve o nível previsto.

O mesmo mecanismo aceita, no lugar do valor hipotético, a chuva **prevista** por
um serviço meteorológico.

In [ ]:
def simular_chuva(linha_features, chuva_mmh, horas_de_chuva=6):
    """
    linha_features : Series com as FEATS do instante atual
    chuva_mmh      : intensidade hipotética (mm/h)
    horas_de_chuva : por quantas horas essa intensidade se mantém
    Devolve uma nova linha de features com a chuva injetada.
    """
    L = linha_features.copy()
    L["chuva"] = chuva_mmh
    for j in (3, 6, 12, 24, 48, 72):
        L[f"chuva_acum{j}"] = L[f"chuva_acum{j}"] + min(j, horas_de_chuva)*chuva_mmh
    L["chuva_max6"] = max(L["chuva_max6"], chuva_mmh)
    if chuva_mmh > 1:
        L["horas_sem_chuva"] = 0
    L["chuva24_x_nivel"] = L["chuva_acum24"] * L["nivel"] / 100.0
    return L


def prever(linha_features, horizonte):
    """Devolve o nível absoluto previsto (nível atual + variação prevista)."""
    d = modelos[horizonte].predict(pd.DataFrame([linha_features])[FEATS])[0]
    return float(linha_features["nivel"] + d)


# cenário de teste: rio baixo, solo seco (o caso mais desfavorável à chuva)
disp = X[FEATS].dropna()
seco = disp[(disp["nivel"] < 80) & (disp["chuva_acum24"] < 1)
            & (disp.index.year == ANO_TESTE)]
linha = seco.iloc[len(seco)//2]
print(f"cenário: nível {linha['nivel']:.0f} cm, sem chuva há "
      f"{linha['horas_sem_chuva']:.0f} h ({seco.index[len(seco)//2]:%d/%m/%Y %H:%M})\n")

grade = [0, 2, 5, 10, 20, 40]
tab = []
for mmh in grade:
    L = simular_chuva(linha, mmh, horas_de_chuva=6)
    tab.append({"chuva (mm/h)": mmh,
                **{f"t+{h}h": round(prever(L, h), 1) for h in HORIZONTES}})
sim = pd.DataFrame(tab)
sim

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.4))
for h, cor in zip(HORIZONTES, ["#17A2C4", "#0B7FAB", "#0A2F4E"]):
    ax.plot(sim["chuva (mm/h)"], sim[f"t+{h}h"], "o-", color=cor,
            label=f"t+{h}h", lw=2)
ax.axhline(linha["nivel"], color="gray", ls="--", lw=1, label="nível atual")
ax.set_xlabel("chuva simulada (mm/h por 6 h)")
ax.set_ylabel("nível previsto (cm)")
ax.set_title("Resposta do modelo à chuva simulada")
ax.legend(frameon=False, fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.grid(color="#EEF3F7"); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

### Limite conhecido da simulação

A resposta **satura** acima de cerca de 10 a 20 mm/h: intensidades muito altas
produzem previsões parecidas. O histórico contém poucas horas de chuva nessa
magnitude, e árvores de decisão não extrapolam além da faixa que viram no treino.

A faixa de 0 a 10 mm/h já produz variação expressiva e é onde a simulação é
apoiada em dados. Acima disso, é extrapolação.

## 7. Previsão vs. real no evento de subida mais rápida

In [ ]:
H = 24
Xtr, ytr, Xte, yte = separar(X, H)
d_pred = pd.Series(modelos[H].predict(Xte), index=Xte.index)
nivel_prev = Xte["nivel"] + d_pred
nivel_real = Xte["nivel"] + yte

pico = yte.idxmax()
print(f"maior subida em {H}h: {yte.max():.0f} cm em {pico:%d/%m/%Y %H:%M}")
jan = slice(pico - pd.Timedelta("30h"), pico + pd.Timedelta("30h"))

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(nivel_real[jan].index, nivel_real[jan].values, lw=2.2, color="#0A2F4E",
        label="real")
ax.plot(nivel_prev[jan].index, nivel_prev[jan].values, lw=1.8, ls="--",
        color="#E8960C", label=f"previsto (t+{H}h)")
ax.plot(Xte["nivel"][jan].index, Xte["nivel"][jan].values, lw=1.2, color="gray",
        alpha=0.8, label="persistência (nível atual)")
for nome, lim in LIMIARES.items():
    ax.axhline(lim, ls=":", lw=1, color="red", alpha=0.5)
ax.set_ylabel("nível (cm)"); ax.legend()
ax.set_title(f"Evento de subida rápida — {pico:%d/%m/%Y %H:%M}")
plt.tight_layout(); plt.show()

## 8. Baixar os modelos

In [ ]:
import json
with open("features_delta.json", "w") as f:
    json.dump({"features": FEATS, "horizontes": list(HORIZONTES),
               "alvo": "variacao_cm", "limiares": LIMIARES}, f, indent=2)

for nome in ([f"modelo_delta_{h}h.json" for h in HORIZONTES]
             + ["metricas_delta.csv", "features_delta.json"]):
    files.download(nome)

---
## Como usar estes modelos

Eles preveem **variação**, não nível:

```python
import xgboost as xgb

m = xgb.XGBRegressor()
m.load_model("modelos/modelo_delta_6h.json")
nivel_previsto = nivel_atual + m.predict(features)[0]
```

O arquivo `features_delta.json` guarda a lista e a **ordem exata** das 23
features — o modelo espera essa ordem.

## Escopo de aplicação

Os modelos aprenderam a cota do rio na estação 61305000, onde o nível varia de
14 a 447 cm e o rio responde em horas. **Não se aplicam à régua urbana de 1 m**,
que mede transbordamento em outra escala e outra dinâmica, nela a projeção do
tempo restante é feita por extrapolação da tendência observada.